#### Setup

In [6]:
"""
Model 1: IRIS CLASSIFIACATION
Pattern: Basic classification pipeline
"""

import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle
import os

os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

In [9]:
def load_data():
    """ 
    PRIMITIVE: load_data 
    load the Iris dataset from sklearn
    """

    # Load the built-in iris dataset
    iris = load_iris()

    # Convert to DataFrame for easier handling
    X = pd.DataFrame(iris.data, columns=iris.feature_names)
    y = pd.Series(iris.target, name='species')

    # Print info
    print(f"Loaded {len(X)} samples")
    print(f"Features: {len(X.columns)}")
    print(f"Classes: {len(np.unique(y))}")
    print(f"Target names: {list(iris.target_names)}")

    return X, y

# Test
X, y = load_data()
print(f"\nFirst 10 rows:\n{X.head()}")


Loaded 150 samples
Features: 4
Classes: 3
Target names: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

First 10 rows:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2


##### Split Data Into Train and Test

In [10]:
def split_data(X, y):
    """
    Primitive: split_data
    Split into train and test sets
    Standard: 80/20 split, fixed random seed
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,    # 20% for testing
        random_state=42,  # Reproducible results
        stratify=y        # Keep class distribution balanced
    )

    print(f"Training samples: {len(X_train)}")
    print(f"Testing samples: {len(X_test)}")
    print(f"\nClass distribution in train:")
    print(y_train.value_counts().sort_index())

    return X_train, X_test, y_train, y_test

# Test it
X_train, X_test, y_train, y_test = split_data(X, y)

Training samples: 120
Testing samples: 30

Class distribution in train:
species
0    40
1    40
2    40
Name: count, dtype: int64


#### Train Model

In [13]:
def train_model(X_train, y_train):
    """
    Primitive: train_model
    Train a random forest classifier
    """
    # Initialise model
    model = RandomForestClassifier(
        n_estimators=100, # Number of trees
        random_state=42,  # Reproducible results
        max_depth=5     # Prevent overfitting
    )

    # Fit the Model
    model.fit(X_train, y_train)

    print("Model training complete")
    print(f"Number of trees: {model.n_estimators}")
    print(f"Feature importances: {model.feature_importances_}")

    return model

# Test
model = train_model(X_train, y_train)

Model training complete
Number of trees: 100
Feature importances: [0.11597203 0.01424578 0.43164136 0.43814083]


#### Evaluate Model

In [14]:
def evaluate_model(model, X_test, y_test):
    """
    Primitive: evaluate model
    Test the model and print metrics
    """

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate accuracy
    accuracy =accuracy_score(y_test, y_pred)

    # Print results
    print(f"Accuracy: {accuracy:.3f}")
    print(f"\nClassifiaction Report:")
    print(classification_report(y_test, y_pred,
                                target_names=["setosa", "versicolor", "virginica"]))
    
    return accuracy

# Test it 
accuracy = evaluate_model(model, X_test, y_test)




Accuracy: 0.933

Classifiaction Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



In [17]:
def save_model(model, filepath='models/iris_model.pkl'):
    """
    Primitive: Save Model
    save trained model to disk
    """
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)

    print(f"Model saved to {filepath}")
    print(f"File size: {os.path.getsize(filepath)} bytes")

# Test 
save_model(model)

Model saved to models/iris_model.pkl
File size: 153716 bytes


#### Load Model

In [19]:
def load_model(filepath='models/iris_model.pkl'):
    """
    Primitive: load_model
    Load a saved model from disk
    """
    with open(filepath, 'rb') as f:
        model = pickle.load(f)

    print(f"Model loaded from {filepath}")

    return model

# Test
loaded_model = load_model()
print(f"Model type: {type(loaded_model)}")

Model loaded from models/iris_model.pkl
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


##### Make Predictions

In [20]:
def predict(model, new_data):
    """
    Primitive: predict
    Make predictions on new data
    """

    predictions = model.predict(new_data)

    # Get prediction probabilities
    probabilities = model.predict_proba(new_data)

    print(f"Predictions: {predictions}")
    print(f"Probabilities: \n{probabilities}")

    return predictions

# Test with sample data
sample = [[5.1, 3.5, 1.4, 0.2]] # Should be satatosa (class 0)
sample_df = pd.DataFrame(sample, columns=X.columns)

prediction = predict(loaded_model, sample_df)
print(f"Actual name:{['satosa', 'versicolor', 'virginica'][prediction[0]]}")

Predictions: [0]
Probabilities: 
[[1. 0. 0.]]
Actual name:satosa
